# 🚀 Zindi – Bénin Olympiad Phase 1 : Détection de routes satellite (Version Améliorée)

**Objectif :** Atteindre un score AUC-ROC de **0.987868** avec une robustesse et une stabilité légendaires.

**Stack :** fastai + timm + Albumentations

## ⚙️ Configuration Globale et Reproductibilité

In [ ]:
import sys
import random, os, warnings, shutil
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime
from fastai.vision.all import *
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
import cv2

# Installation des librairies
!{sys.executable} -m pip install -q --upgrade fastai timm albumentations

SEED        = 42
IMG_SIZE    = 320
BS          = 16
ARCH        = 'convnext_large_in22k'
EPOCHS_A    = 5
EPOCHS_B    = 20
EPOCHS_C    = 10
LR_A        = 3e-3
LR_B        = slice(5e-6, 2e-4)
LR_C        = slice(1e-6, 4e-5)

KAGGLE_ROOT = Path('/kaggle/input/datasets/hilagbag')
IMG_PATH    = KAGGLE_ROOT / 'images-zip/Images/Images'
TRAIN_CSV   = KAGGLE_ROOT / 'images-zip/Train.csv'
TEST_CSV    = KAGGLE_ROOT / 'images-zip/Test.csv'
SAMPLE_SUB  = KAGGLE_ROOT / 'images-zip/SampleSubmission.csv'

OUTPUT_DIR  = Path('./output')
MODEL_DIR   = OUTPUT_DIR / 'models'
SUBMIT_DIR  = OUTPUT_DIR / 'submissions'
ERROR_IMAGES_DIR = OUTPUT_DIR / 'error_images'

for d in [OUTPUT_DIR, MODEL_DIR, SUBMIT_DIR, ERROR_IMAGES_DIR]: d.mkdir(exist_ok=True)

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Config prête sur {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 📊 Chargement des Données

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
train_df['Image_ID'] = train_df['Image_ID'].astype(str)
test_df['Image_ID'] = test_df['Image_ID'].astype(str)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
_, val_idx = list(skf.split(train_df, train_df['Target']))[0]

print(f"Train: {len(train_df)}, Val: {len(val_idx)}")

## ✨ Augmentations de Données Massives

In [ ]:
tfms_light_albu = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=180, p=0.75, border_mode=cv2.BORDER_REFLECT_101),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=15, p=0.75, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.75),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

tfms_heavy_albu = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=180, p=0.9, border_mode=cv2.BORDER_REFLECT_101),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.3, rotate_limit=45, p=0.9, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.9),
    A.GaussNoise(p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class AlbumentationsTransform(ItemTransform):
    def __init__(self, aug): self.aug = aug
    def encodes(self, img: PILImage):
        aug_img = self.aug(image=np.array(img))['image']
        return TensorImage(aug_img)

def get_dls(df, aug_tfms, bs, img_path, valid_idx):
    return ImageDataLoaders.from_df(
        df, path=img_path, suff='.tif', fn_col='Image_ID', label_col='Target',
        item_tfms=[Resize(IMG_SIZE, method='pad'), AlbumentationsTransform(aug_tfms)],
        bs=bs, valid_idx=valid_idx, seed=SEED
    )

dls_light = get_dls(train_df, tfms_light_albu, BS, IMG_PATH, val_idx)
dls_heavy = get_dls(train_df, tfms_heavy_albu, BS, IMG_PATH, val_idx)
print("DataLoaders prêts.")

## 🧠 Entraînement

In [ ]:
save_best_cb = SaveModelCallback(monitor='roc_auc_score', comp=np.greater, fname='best_model_sota')
learn = vision_learner(dls_light, ARCH, metrics=[accuracy, RocAucBinary()], cbs=[save_best_cb], pretrained=True, n_out=2)

print("Phase A...")
learn.fine_tune(EPOCHS_A, freeze_epochs=EPOCHS_A, base_lr=LR_A)

print("Phase B...")
learn.unfreeze()
learn.fit_one_cycle(EPOCHS_B, LR_B)

print("Phase C...")
learn.dls = dls_heavy
learn.fit_one_cycle(EPOCHS_C, LR_C)

learn.load('best_model_sota')
print("Entraînement terminé.")

## 🔍 Analyse des Erreurs

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
losses, idxs = interp.top_losses(k=100)
interp.plot_top_losses(k=9, figsize=(15,15))

error_df = train_df.iloc[val_idx[idxs]].copy()
sampled_df = train_df.drop(val_idx[idxs]).sample(frac=0.1, random_state=SEED)
ft_df = pd.concat([error_df, sampled_df]).reset_index(drop=True)

learn.dls = get_dls(ft_df, tfms_heavy_albu, BS, IMG_PATH, [])
learn.fit_one_cycle(3, slice(1e-7, 1e-6))
learn.load('best_model_sota')
print("Fine-tuning sur erreurs terminé.")

## 🚀 Soumission

In [ ]:
test_dl = learn.dls.test_dl(test_df, with_labels=False)
learn.dls = dls_heavy
preds, _ = learn.tta(dl=test_dl, n=15)

submission_df = pd.DataFrame({'Image_ID': test_df['Image_ID'], 'Target': preds[:, 1].numpy()})
sub_file = SUBMIT_DIR / f"sub_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
submission_df.to_csv(sub_file, index=False)
print(f"Soumission créée: {sub_file}")